# Part 3: Data Analytics

This notebook performs data analysis on:
1. BLS time-series data (from Part 1)
2. Population data from DataUSA API (from Part 2)

## Analysis Tasks

1. Calculate mean and standard deviation of US population (2013-2018)
2. Find best year per series_id (year with max sum of values)
3. Generate combined report for PRS30006032 Q01 with population data


## Setup and Imports


In [1]:
import sys
from pathlib import Path

# Add src to path to import rearc package
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import boto3
import json
from io import StringIO
import numpy as np

# Import analytics queries
from rearc.analytics import (
    query1_population_stats,
    query2_best_year_per_series,
    query3_combined_report
)


## Configuration


In [2]:
# Configuration
# Option 1: Load from S3 (set USE_LOCAL_DATA = False)
# Option 2: Load from local files (set USE_LOCAL_DATA = True)
USE_LOCAL_DATA = False  # Set to True to use local test data

if USE_LOCAL_DATA:
    # Local data paths (for testing without AWS)
    LOCAL_DATA_DIR = project_root / 'data' / 'local'
    BLS_DATA_PATH = LOCAL_DATA_DIR / 'pr.data.0.Current'
    POPULATION_DATA_PATH = LOCAL_DATA_DIR / 'population_data.json'
    print("Using local data files")
else:
    # S3 Configuration - Try to load from config/outputs.json, fallback to hardcoded
    config_file = project_root / 'config' / 'outputs.json'
    if config_file.exists():
        with open(config_file) as f:
            config = json.load(f)
            BUCKET_NAME = config.get('s3_bucket_name', 'rearc-data-pipeline-data-08041c62')
        print(f"✓ Loaded bucket name from config: {BUCKET_NAME}")
    else:
        # Fallback to hardcoded bucket name
        BUCKET_NAME = 'rearc-data-pipeline-data-08041c62'
        print(f"⚠ Config file not found, using hardcoded bucket: {BUCKET_NAME}")
    
    BLS_DATA_KEY = 'pr.data.0.Current'  # BLS time-series data
    
    # Initialize S3 client
    s3_client = boto3.client('s3')
    
    # Function to find the latest population data file
    def find_latest_population_file(bucket_name, s3_client):
        """Find the most recent population_data_*.json file in S3."""
        try:
            # List all objects with population_data_ prefix
            response = s3_client.list_objects_v2(
                Bucket=bucket_name,
                Prefix='population_data_'
            )
            
            if 'Contents' not in response:
                return None
            
            # Filter for .json files and sort by last modified (newest first)
            json_files = [
                obj for obj in response['Contents']
                if obj['Key'].endswith('.json')
            ]
            
            if not json_files:
                return None
            
            # Sort by last modified date (newest first)
            latest = max(json_files, key=lambda x: x['LastModified'])
            return latest['Key']
        except Exception as e:
            print(f"Error finding latest population file: {e}")
            return None
    
    # Find the latest population data file
    POPULATION_DATA_KEY = find_latest_population_file(BUCKET_NAME, s3_client)
    if POPULATION_DATA_KEY:
        print(f"✓ Found latest population data file: {POPULATION_DATA_KEY}")
    else:
        print("⚠ No population data files found in S3")
        POPULATION_DATA_KEY = None  # Will cause error if used
    
    print("Using S3 data")


✓ Loaded bucket name from config: rearc-data-pipeline-dev-data-fe445231
✓ Found latest population data file: population_data_20260126_020032.json
Using S3 data


## Load Data


In [3]:
# Load BLS time-series data
if USE_LOCAL_DATA:
    # Load from local file
    bls_content = BLS_DATA_PATH.read_text()
    bls_df = pd.read_csv(StringIO(bls_content), sep='\t')
else:
    # Load from S3
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=BLS_DATA_KEY)
    bls_content = response['Body'].read().decode('utf-8')
    bls_df = pd.read_csv(StringIO(bls_content), sep='\t')

# Clean column names (remove extra spaces)
bls_df.columns = bls_df.columns.str.strip()

print(f"BLS Data Shape: {bls_df.shape}")
print(f"BLS Data Columns: {bls_df.columns.tolist()}")
print(f"\nFirst few rows:")
bls_df.head()


BLS Data Shape: (37521, 5)
BLS Data Columns: ['series_id', 'year', 'period', 'value', 'footnote_codes']

First few rows:


,series_id,year,period,value,footnote_codes
0,PRS30006011,1995,Q01,2.6,NaN
1,PRS30006011,1995,Q02,2.1,NaN
2,PRS30006011,1995,Q03,0.9,NaN
3,PRS30006011,1995,Q04,0.1,NaN
4,PRS30006011,1995,Q05,1.4,NaN


In [4]:
# Load Population data
if USE_LOCAL_DATA:
    # Load from local file
    population_data = json.loads(POPULATION_DATA_PATH.read_text())
else:
    # Load from S3
    if POPULATION_DATA_KEY is None:
        raise ValueError("No population data file found in S3. Make sure the data sync Lambda has run at least once.")
    
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=POPULATION_DATA_KEY)
    population_content = response['Body'].read().decode('utf-8')
    population_data = json.loads(population_content)

# Convert to DataFrame
population_df = pd.DataFrame(population_data.get('data', []))

print(f"Population Data Shape: {population_df.shape}")
print(f"Population Data Columns: {population_df.columns.tolist()}")
if 'Year' in population_df.columns:
    print(f"\nYears available: {sorted(population_df['Year'].unique())}")
print(f"\nFirst few rows:")
population_df.head(10)


Population Data Shape: (10, 4)
Population Data Columns: ['Nation ID', 'Nation', 'Year', 'Population']

Years available: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023)]

First few rows:


,Nation ID,Nation,Year,Population
0,01000US,United States,2013,316128839.0
1,01000US,United States,2014,318857056.0
2,01000US,United States,2015,321418821.0
3,01000US,United States,2016,323127515.0
4,01000US,United States,2017,325719178.0
5,01000US,United States,2018,327167439.0
6,01000US,United States,2019,328239523.0
7,01000US,United States,2021,331893745.0
8,01000US,United States,2022,333287562.0
9,01000US,United States,2023,334914896.0


## Data Cleaning


In [5]:
# Clean BLS data - trim whitespaces from string columns
string_columns = bls_df.select_dtypes(include=['object']).columns
for col in string_columns:
    if col != 'footnote_codes':  # Skip footnote_codes which may have NaN
        bls_df[col] = bls_df[col].str.strip()

# Ensure proper data types
if 'year' in bls_df.columns:
    bls_df['year'] = pd.to_numeric(bls_df['year'], errors='coerce')
if 'value' in bls_df.columns:
    bls_df['value'] = pd.to_numeric(bls_df['value'], errors='coerce')

print("✓ BLS data cleaned")
print(f"  - Trimmed column names and string values")
print(f"  - Converted year and value to numeric")
print(f"  - Total records: {len(bls_df)}")
print(f"  - Unique series_ids: {bls_df['series_id'].nunique()}")
print(f"  - Year range: {bls_df['year'].min()} - {bls_df['year'].max()}")


✓ BLS data cleaned
  - Trimmed column names and string values
  - Converted year and value to numeric
  - Total records: 37521
  - Unique series_ids: 282
  - Year range: 1995 - 2025


## Query 1: Population Statistics (2013-2018)


In [6]:
# Query 1: Population Statistics (2013-2018)
# Use the analytics module function
result = query1_population_stats(population_df)

print("=" * 60)
print("Query 1: Population Statistics (2013-2018)")
print("=" * 60)
print(f"\nMean Population: {result['mean']:,.0f}")
print(f"Standard Deviation: {result['std_dev']:,.0f}")
print(f"\nFull Result:")
print(result)

# Display the filtered data used for calculation
filtered_pop = population_df[
    (population_df['Year'] >= 2013) & (population_df['Year'] <= 2018)
].sort_values('Year')
print(f"\nData used for calculation ({len(filtered_pop)} years):")
print(filtered_pop[['Year', 'Population']].to_string(index=False))


Query 1: Population Statistics (2013-2018)

Mean Population: 322,069,808
Standard Deviation: 4,158,441

Full Result:
{'mean': 322069808.0, 'std_dev': 4158441.040908095}

Data used for calculation (6 years):
 Year  Population
 2013 316128839.0
 2014 318857056.0
 2015 321418821.0
 2016 323127515.0
 2017 325719178.0
 2018 327167439.0


## Query 2: Best Year per Series ID


In [7]:
# Query 2: Best Year per Series ID
# Use the analytics module function
result_df = query2_best_year_per_series(bls_df)

print("=" * 60)
print("Query 2: Best Year per Series ID")
print("=" * 60)
print(f"\nTotal series analyzed: {len(result_df)}")
print(f"\nFirst 20 results:")
print(result_df.head(20).to_string(index=False))

# Show some statistics
print(f"\nStatistics:")
print(f"  - Year range in results: {result_df['year'].min()} - {result_df['year'].max()}")
print(f"  - Average best year value: {result_df['value'].mean():.2f}")
print(f"  - Max best year value: {result_df['value'].max():.2f}")

# Show example for PRS30006032
prs32_result = result_df[result_df['series_id'] == 'PRS30006032']
if len(prs32_result) > 0:
    print(f"\nExample - PRS30006032:")
    print(f"  Best year: {prs32_result.iloc[0]['year']}")
    print(f"  Sum value: {prs32_result.iloc[0]['value']}")

result_df


Query 2: Best Year per Series ID

Total series analyzed: 282

First 20 results:
  series_id  year   value
PRS30006011  2022  20.500
PRS30006012  2022  17.100
PRS30006013  1998 705.895
PRS30006021  2010  17.700
PRS30006022  2010  12.400
PRS30006023  2014 503.216
PRS30006031  2022  20.500
PRS30006032  2021  17.100
PRS30006033  1998 702.672
PRS30006061  2022  34.500
PRS30006062  2021  29.800
PRS30006063  2024 643.539
PRS30006081  2021  24.500
PRS30006082  2021  24.500
PRS30006083  2022 131.294
PRS30006091  2002  43.400
PRS30006092  2002  44.300
PRS30006093  2013 514.158
PRS30006101  2020  33.100
PRS30006102  2020  35.700

Statistics:
  - Year range in results: 1995 - 2024
  - Average best year value: 204.87
  - Max best year value: 1047.34

Example - PRS30006032:
  Best year: 2021
  Sum value: 17.1


,series_id,year,value
27,PRS30006011,2022,20.500
58,PRS30006012,2022,17.100
65,PRS30006013,1998,705.895
108,PRS30006021,2010,17.700
139,PRS30006022,2010,12.400
...,...,...,...
8459,PRS88003192,2002,282.800
8512,PRS88003193,2024,862.564
8541,PRS88003201,2022,38.900
8572,PRS88003202,2022,29.700


## Query 3: Combined Report (PRS30006032 Q01 + Population)


In [8]:
# Query 3: Combined Report (PRS30006032 Q01 + Population)
# Use the analytics module function
result_df = query3_combined_report(bls_df, population_df)

print("=" * 60)
print("Query 3: Combined Report (PRS30006032 Q01 + Population)")
print("=" * 60)
print(f"\nTotal records: {len(result_df)}")
print(f"Records with population data: {result_df['Population'].notna().sum()}")

# Show all records with population data
result_with_pop = result_df[result_df['Population'].notna()].sort_values('year')
print(f"\nRecords with Population Data ({len(result_with_pop)} records):")
print(result_with_pop.to_string(index=False))

# Show records for 2013-2018 specifically
result_2013_2018 = result_with_pop[
    (result_with_pop['year'] >= 2013) & (result_with_pop['year'] <= 2018)
]
print(f"\nRecords for 2013-2018 ({len(result_2013_2018)} records):")
print(result_2013_2018.to_string(index=False))

# Show all records (including those without population data)
print(f"\nAll Records (including those without population data):")
result_df.sort_values('year')


Query 3: Combined Report (PRS30006032 Q01 + Population)

Total records: 31
Records with population data: 10

Records with Population Data (10 records):
  series_id  year period  value  Population
PRS30006032  2013    Q01    0.5 316128839.0
PRS30006032  2014    Q01   -0.1 318857056.0
PRS30006032  2015    Q01   -1.7 321418821.0
PRS30006032  2016    Q01   -1.4 323127515.0
PRS30006032  2017    Q01    0.9 325719178.0
PRS30006032  2018    Q01    0.5 327167439.0
PRS30006032  2019    Q01   -1.6 328239523.0
PRS30006032  2021    Q01    0.7 331893745.0
PRS30006032  2022    Q01    5.3 333287562.0
PRS30006032  2023    Q01    0.3 334914896.0

Records for 2013-2018 (6 records):
  series_id  year period  value  Population
PRS30006032  2013    Q01    0.5 316128839.0
PRS30006032  2014    Q01   -0.1 318857056.0
PRS30006032  2015    Q01   -1.7 321418821.0
PRS30006032  2016    Q01   -1.4 323127515.0
PRS30006032  2017    Q01    0.9 325719178.0
PRS30006032  2018    Q01    0.5 327167439.0

All Records (includ

,series_id,year,period,value,Population
0,PRS30006032,1995,Q01,0.0,NaN
1,PRS30006032,1996,Q01,-4.2,NaN
2,PRS30006032,1997,Q01,2.8,NaN
3,PRS30006032,1998,Q01,0.9,NaN
4,PRS30006032,1999,Q01,-4.1,NaN
5,PRS30006032,2000,Q01,0.5,NaN
6,PRS30006032,2001,Q01,-6.3,NaN
7,PRS30006032,2002,Q01,-6.6,NaN
8,PRS30006032,2003,Q01,-5.7,NaN
9,PRS30006032,2004,Q01,2.0,NaN


## Summary

All three analytical queries have been successfully executed with the following results:

1. **Query 1: Population Statistics (2013-2018)**
   - Mean Population: **322,069,808**
   - Standard Deviation: **4,158,441**
   - Based on 6 years of data (2013-2018)

2. **Query 2: Best Year per Series ID**
   - **282 series** analyzed
   - For each series_id, identified the year with the maximum sum of quarterly values
   - Year range in results: 1995 - 2024
   - Example: PRS30006032's best year is 2021 with sum value of 17.1

3. **Query 3: Combined Report (PRS30006032 Q01 + Population)**
   - **31 total records** (all years for PRS30006032 Q01)
   - **10 records** with matching population data (years 2013-2019, 2021-2023)
   - Shows quarterly economic indicator values alongside population data where available
   - Correctly handles missing 2020 population data (NaN)


## Findings and Observations

### Data Quality Observations

1. **BLS Data Contains Future Year (2025)**
   - The BLS dataset includes data for year 2025, which appears to be forecast/projection data rather than historical data
   - Year range in BLS data: 1995 - 2025
   - This is visible in Query 3 results where PRS30006032 Q01 shows a value of 0.4 for 2025
   - **Recommendation**: Consider filtering out future years if only historical data is required, or clearly document that forecasts are included

2. **Missing 2020 Population Data**
   - Population dataset is missing data for year 2020
   - Available years: 2013-2019, 2021-2023 (10 total records)
   - This is expected due to COVID-19 data collection challenges during 2020
   - Query 3 correctly handles this with NaN values for 2020

3. **Query Consistency**
   - Query 2 shows PRS30006032's best year is 2021 with sum value of 17.1 (sum of all quarters)
   - Query 3 shows PRS30006032 Q01 value of 0.7 for 2021
   - These are consistent: Query 2 sums all quarters per year, while Query 3 shows individual Q01 values

### Data Quality Summary

✅ **Good**:
- Population growth trend is reasonable and consistent
- No obvious data corruption or outliers
- Missing values handled correctly in joins
- All three queries execute successfully

⚠️ **Consider**:
- Documenting that 2020 population data is unavailable